# Laboratory 01 — The simple harmonic oscillator

In this laboratory you will put isochronism on trial — measure whether the period really
ignores the amplitude — watch energy slosh under a flat total, and then treat the
simulation as a noisy instrument and extract $\omega_0$ from it with an honest error bar.

Work through it in order. Where the notebook asks you to predict, write your prediction in
the cell provided **before** running the next cell.

## Model specification

| | |
|---|---|
| **System** | a point mass $m$ on a massless linear spring of stiffness $k$, in one dimension |
| **Dynamics** | $m\ddot{x} = -kx - b\dot{x}$, closed forms where they exist, velocity-Verlet elsewhere |
| **Boundary** | none — the mass moves on an infinite line and nothing is exchanged |
| **Ensemble** | one deterministic trajectory per $(x_0, v_0)$; measurement noise is Gaussian, independent, seeded |
| **Ignored** | spring mass, nonlinearity at large extension, and every microscopic mechanism behind $b$ |
| **Valid when** | displacements stay in the linear regime and the time step stays well below the period |
| **Failure modes** | amplitudes that feel the nonlinearity; steps near the stability limit; asking where the energy goes |

All the physics lives in `wavelab.oscillators` and `wavelab.measurement` — open them and
read them.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# wavelab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy and matplotlib, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# wavelab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("wavelab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelab import measurement, oscillators
from wavelab.validation import seed_study

MASS = 0.5  # kg
STIFFNESS = 8.0  # N/m
OMEGA0 = oscillators.natural_frequency(MASS, STIFFNESS)
PERIOD = 2 * np.pi / OMEGA0

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"omega0 = {OMEGA0} rad/s   period = {PERIOD:.4f} s")

## Part 1 — The motion

Release the cart from rest at $x_0 = 0.1$ m. Left: position and (scaled) velocity — watch
the quarter-period lag. Right: the same motion drawn in *phase space*, the $(x, v)$ plane,
where the whole history is one closed ellipse.

In [ ]:
t = np.linspace(0.0, 3 * PERIOD, 1200)
x = oscillators.position(t, MASS, STIFFNESS, 0.1, 0.0)
v = oscillators.velocity(t, MASS, STIFFNESS, 0.1, 0.0)

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.6))
left.plot(t, x, label="x (m)")
left.plot(t, v / OMEGA0, label="v/ω₀ (m)")
left.set_xlabel("t (s)"), left.legend()
left.set_title("velocity leads position by a quarter period")
right.plot(x, v, lw=1.2)
right.set_xlabel("x (m)"), right.set_ylabel("v (m/s)")
right.set_title("phase space: the energy ellipse")
plt.tight_layout()
plt.show()

## Part 2 — Isochronism on trial

### Predict

1. Release from $0.05$ m instead of $0.10$ m: does the period change? By how much?
2. Keep the release point but *quadruple the mass*: what happens to the period?
3. Where is the cart moving fastest — at the release point, at the centre, or somewhere in
   between?

Write your prediction **before** running the next cell.

**Your prediction:**

In [ ]:
def fitted_period(mass, stiffness, x0):
    span = 6 * 2 * np.pi / oscillators.natural_frequency(mass, stiffness)
    tt = np.linspace(0.0, span, 2400)
    fit = measurement.fit_cosine(tt, oscillators.position(tt, mass, stiffness, x0, 0.0))
    return 2 * np.pi / fit.omega


amplitudes = [0.02, 0.05, 0.1, 0.2]
periods = [fitted_period(MASS, STIFFNESS, a) for a in amplitudes]
for a, p in zip(amplitudes, periods, strict=True):
    print(f"release amplitude {a:5.2f} m   ->   period {p:.6f} s")

heavy = fitted_period(4 * MASS, STIFFNESS, 0.1)
print(f"quadrupled mass, same release  ->   period {heavy:.6f} s   ({heavy / periods[2]:.3f}x)")

A factor of ten in amplitude does not move the period; a factor of four in mass doubles it.
Both are $T = 2\pi\sqrt{m/k}$ speaking: the amplitude is simply not in the formula.

## Part 3 — The energy exchange

Watch $K$ and $U$ trade while their sum stays flat — and count how many times the kinetic
energy peaks per cycle before reading the title.

In [ ]:
kinetic, potential, total = oscillators.energies(x, v, MASS, STIFFNESS)

plt.figure(figsize=(9, 3.6))
plt.plot(t, kinetic, label="K")
plt.plot(t, potential, label="U")
plt.plot(t, total, lw=2, label="K + U")
plt.xlabel("t (s)"), plt.ylabel("energy (J)"), plt.legend()
plt.title("K peaks twice per cycle — once per centre crossing — under a flat total")
plt.tight_layout()
plt.show()
print(f"relative variation of the total: {(total.max() - total.min()) / total[0]:.2e}")

## Part 4 — Simulation as experiment

The closed form is a luxury of the *linear* oscillator; almost everything later in the
course must be integrated numerically. So: check the integrator against the closed form —
then corrupt the signal with detector noise and *measure* $\omega_0$ back out of it, with
an error bar.

In [ ]:
dt = oscillators.max_stable_dt(MASS, STIFFNESS)
trajectory = oscillators.simulate(MASS, STIFFNESS, 0.1, 0.0, dt, 3000)
exact = oscillators.position(trajectory.times, MASS, STIFFNESS, 0.1, 0.0)
print(f"max |numerical - exact| over 30 periods: {np.max(np.abs(trajectory.positions - exact)):.2e} m")

# The virtual apparatus: the exact signal, corrupted by detector noise, then fitted.
noisy = measurement.add_noise(exact, 0.005, rng)
fit = measurement.fit_cosine(trajectory.times, noisy)

plt.figure(figsize=(9, 3.6))
plt.plot(trajectory.times, noisy, ".", ms=2, alpha=0.4, label="measured (noisy)")
plt.plot(trajectory.times,
         fit.amplitude * np.cos(fit.omega * trajectory.times + fit.phase) + fit.offset,
         lw=1.5, label="cosine fit")
plt.xlabel("t (s)"), plt.ylabel("x (m)"), plt.legend()
plt.tight_layout()
plt.show()
print(f"omega_fit = {fit.omega:.4f} ± {fit.omega_err:.4f} rad/s   vs   sqrt(k/m) = {OMEGA0:.4f}")

## Part 5 — Automated checks

The claims this notebook has made, as executable assertions — the same checks run in the
project's test suite, so the page, the notebook and the library cannot silently drift
apart.

In [ ]:
# 1. Integrator vs closed form at a fine step: better than one part in 10^3 of amplitude.
fine = oscillators.simulate(MASS, STIFFNESS, 0.1, 0.2, PERIOD / 1000, 20_000)
reference = oscillators.position(fine.times, MASS, STIFFNESS, 0.1, 0.2)
scale, _ = oscillators.amplitude_phase(0.1, 0.2, OMEGA0)
assert np.max(np.abs(fine.positions - reference)) / scale < 1e-3

# 2. Energy over 100 periods: bounded oscillation at (omega0 dt)^2 / 4, and no drift.
steps = round(PERIOD / dt)
check = oscillators.simulate(MASS, STIFFNESS, 0.1, 0.0, dt, 100 * steps)
_, _, total_check = oscillators.energies(check.positions, check.velocities, MASS, STIFFNESS)
bound = (OMEGA0 * dt) ** 2 / 4
assert (total_check.max() - total_check.min()) / total_check[0] < 1.5 * bound
drift = abs(total_check[-steps:].mean() - total_check[:steps].mean()) / total_check[0]
assert drift < 1e-7

# 3. The measurement pipeline recovers omega0 within its scatter, across 8 seeds.
tt = np.linspace(0.0, 8 * PERIOD, 1601)
clean = oscillators.position(tt, MASS, STIFFNESS, 0.1, 0.0)
study = seed_study(
    lambda r: measurement.fit_cosine(tt, measurement.add_noise(clean, 0.02, r), OMEGA0).omega,
    n_seeds=8,
)
assert study.agrees_with(OMEGA0, n_sigma=3.0)

print("all checks passed")
print(f"energy wobble {(total_check.max() - total_check.min()) / total_check[0]:.2e}   (bound {bound:.2e})")
print(f"omega measured {study.mean:.4f} ± {study.standard_error:.1e}   vs   {OMEGA0}")

## Part 6 — Explore it yourself

The integrator runs live here, so this panel uses a **Run** button (`interact_manual`):
integration is too slow in the browser kernel to re-run on every slider twitch.

Two experiments worth doing:

1. Give the cart an initial velocity as well as a displacement, and watch what changes in
   the phase-space ellipse — and what does *not* change in the period.
2. Turn the damping up from zero (a preview of module 02) and watch the ellipse become a
   spiral: energy leaving, period barely moving.

In [ ]:
import ipywidgets as widgets


def explore(mass=0.5, stiffness=8.0, x0=0.1, v0=0.0, damping=0.0):
    step = oscillators.max_stable_dt(mass, stiffness)
    run = oscillators.simulate(mass, stiffness, x0, v0, step, 4000, damping=damping)
    _, _, total_e = oscillators.energies(run.positions, run.velocities, mass, stiffness)

    fig, (left, middle, right) = plt.subplots(1, 3, figsize=(12, 3.4))
    left.plot(run.times, run.positions, lw=1.0)
    left.set_xlabel("t (s)"), left.set_ylabel("x (m)"), left.set_title("position")
    middle.plot(run.positions, run.velocities, lw=0.8)
    middle.set_xlabel("x (m)"), middle.set_ylabel("v (m/s)"), middle.set_title("phase space")
    right.plot(run.times, total_e, lw=1.2)
    right.set_xlabel("t (s)"), right.set_ylabel("E (J)"), right.set_title("total energy")
    plt.tight_layout()
    plt.show()
    print(f"omega0 = {oscillators.natural_frequency(mass, stiffness):.3f} rad/s")


widgets.interact_manual(
    explore,
    mass=widgets.FloatSlider(min=0.1, max=4.0, step=0.1, value=0.5, description="m (kg)"),
    stiffness=widgets.FloatSlider(min=1.0, max=32.0, step=0.5, value=8.0, description="k (N/m)"),
    x0=widgets.FloatSlider(min=-0.2, max=0.2, step=0.01, value=0.1, description="x0 (m)"),
    v0=widgets.FloatSlider(min=-0.5, max=0.5, step=0.05, value=0.0, description="v0 (m/s)"),
    damping=widgets.FloatSlider(min=0.0, max=2.0, step=0.05, value=0.0, description="b (kg/s)"),
);

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "01-sho.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. The model's spring is perfectly linear. Name one observation from today that would
   change if it were not — and one that would survive.
3. Explain, without equations, why the period ignores the amplitude for this force law —
   and why that argument fails for a pendulum at large angles.

**Your answers:**

1.
2.
3.